Spatial Transformer Testset Evaluation

In [30]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import zarr
import tqdm
from torch.utils.data import Dataset, DataLoader
import threading
import queue
from numcodecs import Blosc
import shutil
import dask.array as da


In [37]:
# Fügen Sie hier die Definition Ihrer UNet3D-Klasse ein.
# Dies ist nur ein einfacher Platzhalter, damit das Skript lauffähig ist.
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        
        # Initialize final layer to predict zero displacement
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

# Fügen Sie hier die Definition Ihrer SpatialTransformer3D-Klasse ein.
# Dies ist eine Standard-Implementierung.
class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super(SpatialTransformer3D, self).__init__()
        self.size = size # Erwartete Größe: (D, H, W)
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids)
        grid = torch.unsqueeze(grid, 0)
        grid = grid.type(torch.FloatTensor)
        self.register_buffer('grid', grid)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]

        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)

        new_locs = new_locs.permute(0, 2, 3, 4, 1)
        new_locs = new_locs[..., [2, 1, 0]]

        return F.grid_sample(src, new_locs, align_corners=True, mode='bilinear')

# Fügen Sie hier Ihre Worker-Funktion zum Speichern ein.
def save_to_zarr_worker(q, warped_zarr, dvf_zarr):
    """
    Ein Worker, der Daten aus einer Queue liest und in Zarr-Arrays schreibt.
    """
    while True:
        data = q.get()
        if data is None: # Stopp-Signal
            break
        
        time_idx, warped_np, dvf_np = data
        
        # Schreibe die 3D-Daten an den richtigen Zeitschlitz im 4D/5D-Array
        try:
            warped_zarr[..., time_idx] = warped_np
            dvf_zarr[..., time_idx, :] = dvf_np
        except Exception as e:
            print(f"Fehler im Writer-Thread beim Index {time_idx}: {e}")
        
        q.task_done()

In [38]:
class SelfRegistrationDataset(Dataset):
    """
    Ein Dataset, das jeden Frame einer 4D-Serie gegen einen festen
    Referenzframe aus derselben Serie lädt.
    """
    def __init__(self, moving_path, fixed_index=0, k=16):
        print(f"Lade 4D Zarr-Array von: {moving_path}")
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_index = fixed_index
        self.k = k

        # Annahme: Form ist (H, W, D, T)
        self.original_shape = self.moving_arr.shape[:-1] # (H, W, D)
        self.num_time_points = self.moving_arr.shape[-1]

        # --- ERSATZ FÜR DivisiblePad ---
        self.padded_shape = self._calculate_padded_shape(self.original_shape)
        self.padding_dims = self._calculate_padding_dims()
        # -----------------------------

        print(f"Originale 3D-Form: {self.original_shape}")
        print(f"Gepaddete 3D-Form: {self.padded_shape}")

        # Lade das feste Referenzbild einmal, padde es und speichere es als Tensor
        print(f"Lade und bereite festes Referenzbild (Index {self.fixed_index}) vor...")
        fixed_np = self.moving_arr[..., self.fixed_index].astype(np.float32) # (H, W, D)
        fixed_np_padded = np.pad(fixed_np, self.padding_dims, mode='constant', constant_values=0)
        
        # Konvertiere zu Tensor (C, D, H, W) für das Modell
        self.fixed_tensor = torch.from_numpy(fixed_np_padded.transpose(2, 0, 1)).unsqueeze(0)

    def _calculate_padded_shape(self, shape):
        return tuple([(s + self.k - 1) // self.k * self.k for s in shape])

    def _calculate_padding_dims(self):
        pads = []
        for i in range(3): # H, W, D
            orig_s = self.original_shape[i]
            pad_s = self.padded_shape[i]
            total_pad = pad_s - orig_s
            pad_before = total_pad // 2
            pad_after = total_pad - pad_before
            pads.append((pad_before, pad_after))
        return tuple(pads)

    def __len__(self):
        return self.num_time_points

    def __getitem__(self, idx):
        # Lade das bewegte Bild für den aktuellen Index
        moving_np = self.moving_arr[..., idx].astype(np.float32)
        moving_np_padded = np.pad(moving_np, self.padding_dims, mode='constant', constant_values=0)
        
        # Konvertiere zu Tensor (C, D, H, W)
        moving_tensor = torch.from_numpy(moving_np_padded.transpose(2, 0, 1)).unsqueeze(0)
        
        return moving_tensor, self.fixed_tensor

In [39]:
DATASET_NAME = '159269_B1'
PART = '14'

In [40]:
# Define paths (please adjust if needed)
dicom_folder = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/DICOM' 
zarr_file = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/{DATASET_NAME}_{PART}.zarr'
file_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/Raw_Datasets/{DATASET_NAME}/{PART}/AIF_2.txt'
results_mococo_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}_warped/'
results_warped_path = f'/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/MRI-Datasets/mdreg_motion_correction_results/{DATASET_NAME}_{PART}_dvs/'

In [41]:
if __name__ == '__main__':
    # --- Konfiguration ---
    MODEL_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/ready_to_use/best_supervised_model_250810_02_batch8_LR1e-5_Ep100_VSpl1_RegWeight01.pth"
    
    # --- KORREKTUR: Pfade klar definieren ---
    # 1. Der Pfad zu Ihrer ursprünglichen, un-umgeformten Zarr-Datei
    original_zarr_file = zarr_file# BITTE ANPASSEN

    # 2. Ein temporärer Pfad, um das umgeformte 4D-Array zu speichern
    reshaped_zarr_file = "temp_reshaped_4d_data.zarr" 

    # 3. Pfade für die finalen Ergebnisse
    results_warped_path = "path/to/output/warped.zarr" # BITTE ANPASSEN
    results_mococo_path = "path/to/output/dvf.zarr"    # BITTE ANPASSEN

    # Interne Variablenzuweisung
    OUTPUT_WARPED_PATH = results_warped_path
    OUTPUT_DVF_PATH = results_mococo_path
    FIXED_IMAGE_INDEX = 1
    NUM_WRITER_THREADS = 8
    
    # --- Schritt 1: Forme das ursprüngliche Array in 4D (H, W, D, T) um ---
    print("Schritt 1: Forme das ursprüngliche Array in 4D (H, W, D, T) um...")
    
    # Lade das ursprüngliche Array als Dask-Array
    original_dask_array = da.from_zarr(original_zarr_file)

    # Definiere die Zieldimensionen
    H, W = 256, 256
    D = 50 
    T = 250 
    assert D * T == original_dask_array.shape[0], "Die Dimensionen passen nicht zusammen!"

    # Führe Reshape und Transpose durch
    reshaped_array = original_dask_array.reshape(T, D, H, W)
    final_4d_array = reshaped_array.transpose(2, 3, 1, 0)

    # Speichere das korrekte 4D-Array temporär
    print(f"Speichere korrekt geformtes 4D-Array nach '{reshaped_zarr_file}'...")
    final_4d_array.to_zarr(reshaped_zarr_file, overwrite=True)
    print("Umformung abgeschlossen.")

    # --- Schritt 2: Setup ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nVerwende Gerät: {device}")
    
    for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
        if os.path.exists(path):
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
            print(f"Warnung: Bestehende Datei/Verzeichnis {path} wurde gelöscht.")

    # --- Schritt 3: Daten und Modell laden ---
    print("\nLade Datensatz aus dem korrekt geformten 4D-Array...")
    
    # --- KORREKTUR: Verwende das neu erstellte, umgeformte Zarr-Array ---
    dataset = SelfRegistrationDataset(reshaped_zarr_file, fixed_index=FIXED_IMAGE_INDEX)
    
    padded_input_size = (dataset.padded_shape[2], dataset.padded_shape[0], dataset.padded_shape[1]) # (D, H, W)
    
    print("\nLade trainiertes Modell...")
    model = UNet3D().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    try:
        model = torch.compile(model)
        print("Modell erfolgreich mit torch.compile() optimiert.")
    except Exception: 
        print("torch.compile() nicht verfügbar oder fehlgeschlagen.")
    stn = SpatialTransformer3D(size=padded_input_size).to(device)
    
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=max(1, os.cpu_count() // 2), pin_memory=True, prefetch_factor=2)

    # --- Schritt 4: Output-Dateien und Writer-Threads erstellen ---
    print("\nErstelle thread-sichere Ausgabedateien...")
    synchronizer = zarr.ThreadSynchronizer()
    compressor = None
    
    original_4d_shape = dataset.original_shape + (dataset.num_time_points,)
    original_4d_chunks = dataset.moving_arr.chunks

    warped_output_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w', shape=original_4d_shape, chunks=original_4d_chunks, dtype=dataset.moving_arr.dtype, compressor=compressor, synchronizer=synchronizer)

    dvf_shape = original_4d_shape + (3,)
    dvf_chunks = original_4d_chunks + (3,)

    dvf_output_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w', shape=dvf_shape, chunks=dvf_chunks, dtype='float32', compressor=compressor, synchronizer=synchronizer)
    print(f"Korrekte DVF-Array-Form erstellt: {dvf_output_zarr.shape}")

    data_queue = queue.Queue(maxsize=NUM_WRITER_THREADS * 8)
    writer_threads = []
    print(f"Starte {NUM_WRITER_THREADS} asynchrone Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS):
        thread = threading.Thread(target=save_to_zarr_worker, args=(data_queue, warped_output_zarr, dvf_output_zarr))
        thread.daemon = True
        thread.start()
        writer_threads.append(thread)

    # --- Schritt 5: Inferenz-Schleife ---
    print(f"\nStarte optimierte Inferenz für {len(dataset)} Volumen...")
    original_shape_dims = dataset.original_shape
    padded_shape_dims = dataset.padded_shape
    model.eval()
    with torch.no_grad():
        for i, (moving_batch, fixed_batch) in enumerate(tqdm.tqdm(dataloader, desc="Verarbeite Testdatensatz")):
            moving_batch = moving_batch.to(device, non_blocking=True)
            fixed_batch = fixed_batch.to(device, non_blocking=True)
            
            with torch.autocast(device_type=str(device), dtype=torch.float16):
                predicted_dvf_batch = model(fixed_batch, moving_batch)
                warped_batch = stn(moving_batch, predicted_dvf_batch)
            
            warped_tensor = warped_batch.squeeze(0).cpu()
            disp_field = predicted_dvf_batch.squeeze(0).cpu()

            pad_h_start = (padded_shape_dims[0] - original_shape_dims[0]) // 2
            pad_w_start = (padded_shape_dims[1] - original_shape_dims[1]) // 2
            pad_d_start = (padded_shape_dims[2] - original_shape_dims[2]) // 2
            
            cropped_warped = warped_tensor[pad_d_start:pad_d_start+original_shape_dims[2], 
                                           pad_h_start:pad_h_start+original_shape_dims[0], 
                                           pad_w_start:pad_w_start+original_shape_dims[1]]
            
            cropped_disp = disp_field[:, 
                                      pad_d_start:pad_d_start+original_shape_dims[2], 
                                      pad_h_start:pad_h_start+original_shape_dims[0], 
                                      pad_w_start:pad_w_start+original_shape_dims[1]]
            
            warped_np = cropped_warped.numpy().transpose(1, 2, 0)
            dvf_np = cropped_disp.numpy().transpose(2, 3, 1, 0)
            
            data_queue.put((i, warped_np, dvf_np))
            
    # --- Schritt 6: Aufräumen ---
    print("\nHauptprozess abgeschlossen. Sende Stopp-Signal an Writer-Threads...")
    for _ in range(NUM_WRITER_THREADS): 
        data_queue.put(None)
    
    print("Warte, bis alle Writer-Threads ihre Arbeit beendet haben...")
    for thread in writer_threads: 
        thread.join()

    print(f"\nVerarbeitung und Speicherung in {OUTPUT_WARPED_PATH} und {OUTPUT_DVF_PATH} abgeschlossen.")

Schritt 1: Forme das ursprüngliche Array in 4D (H, W, D, T) um...
Speichere korrekt geformtes 4D-Array nach 'temp_reshaped_4d_data.zarr'...
Umformung abgeschlossen.

Verwende Gerät: cpu

Lade Datensatz aus dem korrekt geformten 4D-Array...
Lade 4D Zarr-Array von: temp_reshaped_4d_data.zarr
Originale 3D-Form: (256, 256, 50)
Gepaddete 3D-Form: (256, 256, 64)
Lade und bereite festes Referenzbild (Index 1) vor...

Lade trainiertes Modell...
Modell erfolgreich mit torch.compile() optimiert.

Erstelle thread-sichere Ausgabedateien...


AttributeError: module 'zarr' has no attribute 'ThreadSynchronizer'

In [5]:
import dask.array as da

In [6]:
raw_darr = da.from_zarr('MRI-Datasets/DCE')
raw_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000), dtype=float32, chunksize=(256, 256, 1, 1), chunktype=numpy.ndarray>

In [7]:
coreg_darr = da.from_zarr('MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr')
coreg_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000), dtype=float32, chunksize=(256, 256, 1, 1), chunktype=numpy.ndarray>

In [8]:
transfo_darr = da.from_zarr('MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr')
transfo_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000, 3), dtype=float32, chunksize=(256, 256, 1, 1, 3), chunktype=numpy.ndarray>

In [34]:
disp_darr = da.from_zarr('/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/MRI-Datasets/SpatialTransformer_2/warped_results_model_test3.zarr')
disp_darr

dask.array<from-zarr, shape=(256, 256, 50, 1000), dtype=float32, chunksize=(256, 256, 1, 1), chunktype=numpy.ndarray>

In [35]:
warped_darr = da.from_zarr('DCE_codeset/MRI-Datasets/SpatialTransformer_2/warped_results_model_test2.zarr')
warped_darr

FileNotFoundError: file://DCE_codeset/MRI-Datasets/SpatialTransformer_2/warped_results_model_test2.zarr

In [36]:
import cupy as cp
import helpers

In [37]:
slice = 27

In [38]:
raw = raw_darr[:,:,slice,:].compute()
raw = cp.transpose(raw, [2,1,0])

In [39]:
coreg = coreg_darr[:,:,slice,:].compute()
coreg = cp.transpose(coreg, [2,1,0])

In [ ]:
warped = warped_darr[:,:,slice,:].compute()
warped = cp.transpose(warped,[2,1,0])

In [ ]:
transfo = transfo_darr[:,:,slice,:,0]
transfo = cp.transpose(transfo, [2,1,0])

In [42]:
disp = disp_darr[:,:,slice,:].compute()
disp = cp.transpose(disp, [2,1,0])

In [ ]:
helpers.explore_3D_array_comparison_and_diff(coreg, disp, 'twilight')

interactive(children=(IntSlider(value=500, description='Slice:', max=999), Output()), _dom_classes=('widget-in…

In [44]:
helpers.explore_3D_array_comparison_and_diff(raw, warped)

NameError: name 'warped' is not defined

In [22]:
helpers.explore_3D_array_comparison_and_diff(transfo, disp)

NameError: name 'disp' is not defined

In [23]:
disp_darr.to_zarr('/media/shooty/Data/MRI_Data/SpatialTransformer_3/disp_fields')

NameError: name 'disp_darr' is not defined